# Day 27: Combining Pandas with SQL

## From DataFrame to SQL Table
In real projects, data often needs to move between pandas (for analysis) and a proper database (for storage, sharing, or querying at scale). The `to_sql` method takes a DataFrame and writes it directly into a database table, creating the table and its columns automatically based on the DataFrame's structure - you don't have to write any `CREATE TABLE` statements by hand. This is useful when you've cleaned or transformed data in pandas and now want to persist it, or hand it off to a SQL-based tool. In the code below, we create an in-memory SQLite database (it exists only for the session and disappears when the connection closes, which is handy for practice and testing) and push a small `products` DataFrame into a table called `products`. Once run, there's no visible output, but the table now exists inside `conn` and can be queried like any real database table.

In [ ]:
import sqlite3
import pandas as pd

# Create an in-memory SQLite database - lives only in RAM for this session,
# useful for demos/tests where you don't want a file left on disk
conn = sqlite3.connect(':memory:')
df = pd.DataFrame({
    'product': ['Laptop', 'Mouse', 'Keyboard', 'Monitor'],
    'price': [800, 20, 45, 300],
    'stock': [15, 200, 120, 30]
})
# Write the DataFrame into a new SQL table named 'products'.
# if_exists='replace' drops and recreates the table if it already exists,
# so re-running this cell won't cause duplicate-table errors.
df.to_sql('products', conn, index=False, if_exists='replace')

## Querying Back into Pandas
Once data lives in a SQL table, `pd.read_sql_query` lets you run any SQL query against it and get the result back as a DataFrame, ready for further analysis, cleaning, or plotting with the usual pandas tools. This is the pattern you'll use constantly in real jobs: the raw data sits in a database, and you pull out only the slice you need with SQL rather than loading the entire table into memory. The query below selects all columns from `products` but filters to only rows where `price` is greater than 40, using standard SQL `WHERE` syntax. The printed output is a normal pandas DataFrame - notice the `Mouse` row (price 20) is excluded from the result because it fails the filter condition.

In [ ]:
# Query the SQL table with a WHERE filter; the result comes back as a DataFrame
result = pd.read_sql_query('SELECT * FROM products WHERE price > 40', conn)
print(result)

## Mixing SQL Filtering with Pandas Analysis
A very common real-world workflow is to let SQL do the heavy lifting - filtering, joining across tables, or aggregating millions of rows inside the database engine, which is much faster than doing it in Python - and then hand the smaller, already-filtered result to pandas for statistics, visualization, or modeling. Here we query only the products with `stock < 50` (i.e. items running low), which trims the dataset down to just the rows we care about before analyzing it. Calling `.describe()` on that result then gives quick summary statistics (count, mean, standard deviation, min, max, and quartiles) for the numeric columns `price` and `stock`, which is a fast way to sanity-check a subset of data without looking at every row individually.

In [ ]:
# Let SQL filter down to low-stock items first, then use pandas for summary stats
low_stock = pd.read_sql_query('SELECT * FROM products WHERE stock < 50', conn)
print(low_stock.describe())